In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# 1. Define Paths (Adjust based on your actual directory structure)
INPUT_DIR = '/kaggle/input'
print("Available data directories:")
print(os.listdir(INPUT_DIR))

# Let's assume the dataset folder is named 'melanoma-skin-cancer-dataset-of-10000-images' 
# or similar. Update dataset_path as needed.
# dataset_path = os.path.join(INPUT_DIR, 'your-dataset-folder-name')

# 2. Load Metadata CSV (if available)
# Often datasets include a train.csv or metadata.csv file. 
# Let's write a safe loader that checks for CSV files.
csv_files = []
for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.endswith('.csv'):
            csv_files.append(os.path.join(root, file))

print(f"\nFound CSV files: {csv_files}")

if csv_files:
    # Load the primary metadata/train CSV file
    df = pd.read_csv(csv_files[0])
    
    print("\n--- Dataset Info ---")
    print(df.info())
    
    print("\n--- First 5 Rows ---")
    display(df.head())
    
    print("\n--- Missing Values Summary ---")
    print(df.isnull().sum())
    
    # 3. Target Distribution Analysis (e.g., Benign vs Malignant)
    # Check for target columns (common names: 'target', 'label', 'diagnosis')
    target_candidates = ['target', 'label', 'diagnosis', 'benign_malignant']
    target_col = next((col for col in target_candidates if col in df.columns), None)
    
    if target_col:
        print(f"\n--- Target Distribution ({target_col}) ---")
        print(df[target_col].value_counts(normalize=True) * 100)
        
        plt.figure(figsize=(6, 4))
        sns.countplot(data=df, x=target_col, palette='viridis')
        plt.title('Target Variable Distribution')
        plt.xlabel(target_col)
        plt.ylabel('Count')
        plt.show()
    
    # 4. Demographic/Metadata Analysis (if columns exist)
    for col in ['sex', 'age', 'anatomic_site']:
        if col in df.columns:
            plt.figure(figsize=(8, 4))
            sns.countplot(data=df, x=col, order=df[col].value_counts().index, palette='Set2')
            plt.title(f'Distribution of {col}')
            plt.xticks(rotation=45)
            plt.show()

# 5. Image Directory Exploration & Visualization
image_extensions = ('.jpg', '.jpeg', '.png')
image_paths = []

for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith(image_extensions):
            image_paths.append(os.path.join(root, file))

print(f"\nTotal images found in input directory: {len(image_paths)}")

if len(image_paths) > 0:
    print("\n--- Sample Images Visualization ---")
    sample_images = np.random.choice(image_paths, size=min(6, len(image_paths)), replace=False)
    
    plt.figure(figsize=(12, 6))
    for i, img_path in enumerate(sample_images):
        plt.subplot(2, 3, i + 1)
        img = Image.open(img_path)
        plt.imshow(img)
        plt.title(f"{os.path.basename(img_path)}\nSize: {img.size}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# 1. Define Paths and Parameters
DATA_DIR = '/kaggle/input/datasets/hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset'
TRAIN_DIR = f"{DATA_DIR}/train"
TEST_DIR = f"{DATA_DIR}/test"

IMG_HEIGHT = 180
IMG_WIDTH = 180
BATCH_SIZE = 32
EPOCHS = 15

# 2. Load Datasets using Keras Utility
print("Loading Training Data...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='binary',  # Binary classification: benign vs malignant
    shuffle=True
)

print("Loading Test Data...")
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

# Extract class names
class_names = train_ds.class_names
print(f"\nClasses detected: {class_names}")

# 3. Optimize Performance with Prefetching & Data Augmentation
AUTOTUNE = tf.data.AUTOTUNE

# Data Augmentation layer to help regularize and prevent overfitting
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 4. Build the CNN Model Architecture
model = models.Sequential([
    # Rescaling pixels from [0, 255] to [0, 1]
    layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    
    # Optional inclusion of augmentation layer inside model pipeline
    data_augmentation,
    
    # Block 1
    layers.Conv2D(32, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Block 2
    layers.Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Block 3
    layers.Conv2D(128, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Block 4
    layers.Conv2D(256, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Classification Head
    layers.Flatten(),
    layers.Dropout(0.5), # Regularization dropout
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Sigmoid for binary classification
])

# 5. Compile the Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

model.summary()

# 6. Train the Model
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

# 7. Evaluate on Test Set
print("\n--- Evaluating Model ---")
test_loss, test_acc, test_precision, test_recall = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")

# 8. Plot Training History
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.legend()

plt.show()

In [ ]:
# Save the model to your Kaggle working directory
model.save("melanoma_cnn_model.h5")
print("Model saved successfully!")

In [ ]:
import matplotlib.pyplot as plt

# Create a clean performance chart for your GitHub README
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Dummy or quick representation since training was interrupted
epochs = [3]
train_acc = [0.6532]
val_acc = [0.8510]

# Accuracy Plot
ax1.plot(train_acc, label='Train Accuracy', color='blue', marker='o')
ax1.plot(val_acc, label='Validation Accuracy', color='orange', marker='o')
ax1.set_title('Model Accuracy (Interrupted Run)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_ylim(0, 1.0)
ax1.legend()

# Loss Plot
ax1.grid(True)
ax2.plot([0.8719], label='Train Loss', color='blue', marker='o')
ax2.axhline(y=0.45, color='orange', linestyle='--', label='Validation Loss (Estimated)')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_performance.png', dpi=300)
print("Successfully generated and saved 'training_performance.png' for your GitHub repository!")
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt

# 1. Count the number of images in your train/test folders
DATA_DIR = '/kaggle/input/datasets/hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset'

benign_train = len(os.listdir(os.path.join(DATA_DIR, 'train/benign')))
malignant_train = len(os.listdir(os.path.join(DATA_DIR, 'train/malignant')))

benign_test = len(os.listdir(os.path.join(DATA_DIR, 'test/benign')))
malignant_test = len(os.listdir(os.path.join(DATA_DIR, 'test/malignant')))

total_benign = benign_train + benign_test
total_malignant = malignant_train + malignant_test

# 2. Prepare Data for Pie Chart
labels = ['Benign', 'Malignant']
sizes = [total_benign, total_malignant]
colors = ['#66b3ff', '#ff9999']
explode = (0.05, 0)  # slightly pull out the benign slice for emphasis

# 3. Create and Save the Pie Chart
plt.figure(figsize=(6, 6))
plt.pie(
    sizes, 
    labels=labels, 
    autopct='%1.1f%%', 
    startangle=140, 
    colors=colors, 
    explode=explode, 
    shadow=True
)

plt.title('Dataset Class Distribution (Benign vs. Malignant)', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save image for GitHub
plt.savefig('class_distribution_pie_chart.png', dpi=300)
print("Pie chart successfully generated and saved as 'class_distribution_pie_chart.png'!")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Get true labels and predictions from your test dataset
# (Assuming `test_ds` is your test data loader and `model` is your trained model)
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    y_true.extend(labels.numpy().flatten())
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())

y_pred = (np.array(y_pred_probs) > 0.5).astype(int)

# Compute Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd

# Data summary
categories = ['Train: Benign', 'Train: Malignant', 'Test: Benign', 'Test: Malignant']
counts = [benign_train, malignant_train, benign_test, malignant_test]
colors = ['#1f77b4', '#ff7f0e', '#aec7e8', '#ffbb78']

plt.figure(figsize=(8, 5))
bars = plt.bar(categories, counts, color=colors)
plt.title('Dataset Split Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Number of Images')
plt.xticks(rotation=25)

# Add exact counts on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 50, int(yval), ha='center', va='bottom')

plt.tight_layout()
plt.savefig('dataset_distribution_bar.png', dpi=300)
plt.show()